# Bangla Multimodal Fake News: Preprocessing

This notebook prepares the BTMD dataset. It does **not** train any model, so a CPU runtime is enough.

Steps: mount Drive, unpack the code, check the code version, run preflight, build the unified table, make the group split, and run sanity checks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Unpack the project code

The unpack cell always removes the old copy first, so stale code cannot run.

In [ ]:
import shutil
import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")

candidates = list(DRIVE_ROOT.glob("Bangla-Multimodal-FakeNews*.zip")) + list(
    (DRIVE_ROOT / "BFND-data").glob("Bangla-Multimodal-FakeNews*.zip")
)
assert candidates, "Code zip not found on Drive. Upload Bangla-Multimodal-FakeNews.zip first."
ZIP = max(candidates, key=lambda p: p.stat().st_mtime)
print("Using zip:", ZIP)

shutil.rmtree(CODE_DIR, ignore_errors=True)   # remove any old copy so stale code cannot run
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
print("Files in zip:", len(zipfile.ZipFile(ZIP).namelist()))

In [ ]:
import os
from pathlib import Path

os.chdir(CODE_DIR)
config_text = Path("configs/config.py").read_text(encoding="utf-8")
assert "CLIP_IMAGE_MODEL" in config_text, (
    "Old code detected. Upload the latest zip to Drive, then run the unpack cell again."
)
print("Code version OK. Working directory:", os.getcwd())

## 2. Install packages

Torch is already installed in Colab, so it is not installed here.

In [ ]:
!pip install -q "pandas>=2.0" "scikit-learn>=1.3" "pyarrow>=14.0" "openpyxl>=3.1" "rank-bm25>=0.2.2" joblib pytest

## 3. Run the unit tests (no data needed)

In [ ]:
!python -m pytest -q tests/

## 4. Preflight

Checks model IDs, schema, data files and the GPU. A GPU WARN is expected in this notebook. Any FAIL must be fixed before continuing.

In [ ]:
!python scripts/00_preflight.py

## 5. Build the unified table and the group split

In [ ]:
!python scripts/01_prepare_data.py

## 6. Data statistics

In [ ]:
import json
import pandas as pd
from configs.config import PROCESSED_DIR, RAW_DIR

stats = json.loads((PROCESSED_DIR / "data_stats.json").read_text(encoding="utf-8"))
for key, value in stats.items():
    print(f"{key}: {value}")

## 7. Label mapping check

Each raw label should map to the expected project label.

In [ ]:
unified = pd.read_csv(PROCESSED_DIR / "unified.csv")
print(pd.crosstab(unified["label_raw"], unified["label"], margins=True))

## 8. Split check

Every split should contain all three classes.

In [ ]:
split = pd.read_csv(PROCESSED_DIR / "unified_split.csv")
print(pd.crosstab(split["split"], split["label"], margins=True))

## 9. Manual sample check

Read the 20 texts and check that each label looks right. Note any broken or suspicious rows.

In [ ]:
pd.set_option("display.max_colwidth", 120)
print(unified.sample(20, random_state=7)[["sample_id", "label_raw", "label", "has_image", "text"]].to_string())

## 10. Confound checks

Compare text length and source across classes. If they differ a lot, a model may learn the shortcut instead of the content.

In [ ]:
unified["n_chars"] = unified["text"].str.len()
print("Text length (characters) by class:")
print(unified.groupby("label")["n_chars"].describe())

raw = pd.read_parquet(RAW_DIR / "data" / "metadata.parquet")
print("\nSource category by raw label:")
print(pd.crosstab(raw["multi_category"], raw["source_category"], margins=True))

Next: open `02_model_training.ipynb` with a GPU runtime.